セル1: ライブラリのインポートとデータの読み込み

In [ ]:
import pandas as pd

# ファイル名の指定
input_filename = '換算結果_summary_all_features_列入れ替え削除(手作業).csv'

# CSVファイルの読み込み
df = pd.read_csv(input_filename, encoding='utf-8')

# 読み込んだデータの最初の5行を確認
display(df.head())

セル2: データの処理（最初の4列は除外し、5列目以降を集計）

In [ ]:
# 最初の4列（そのまま1行目の値を抽出する列）と、それ以降の列（計算する列）を分ける
first_4_cols = df.columns[:4].tolist()
calc_cols = df.columns[4:].tolist()

# インデックスを10で割った商を利用して10行ごとにグループ化
group_ids = df.index // 10

# 最初の4列は、各グループの最初の行（1行目）の値を取得
df_first = df[first_4_cols].groupby(group_ids).first()

# 残りの列（5列目以降）は、平均と標準偏差を計算
df_calc = df[calc_cols]
df_mean = df_calc.groupby(group_ids).mean()
df_std = df_calc.groupby(group_ids).std().fillna(0) # 1データしかない場合のエラー回避

# 新しい列名リストを作成（最初はそのままの列名、以降は平均・標準偏差上・標準偏差下）
new_cols = first_4_cols.copy()
for col in calc_cols:
    new_cols.extend([f'{col}_平均', f'{col}_標準偏差上', f'{col}_標準偏差下'])

# 結果を格納する空のデータフレームを作成
df_result = pd.DataFrame(index=df_mean.index, columns=new_cols)

# 最初の4列の値を代入
for col in first_4_cols:
    df_result[col] = df_first[col]

# 5列目以降に対して「平均」「標準偏差上」「標準偏差下」を計算して代入
for col in calc_cols:
    mean_series = df_mean[col]
    std_series = df_std[col]
    
    df_result[f'{col}_平均'] = mean_series
    df_result[f'{col}_標準偏差上'] = mean_series + std_series
    df_result[f'{col}_標準偏差下'] = mean_series - std_series

# 処理結果の最初の5行を確認
display(df_result.head())

セル3: 新しいファイルへの出力

In [ ]:
# 出力するファイル名の指定
output_filename = '10行ごと集計結果_一部計算除外.csv'

# utf-8-sig (BOM付きUTF-8)で保存すると、Excel等で開いた際の文字化けを防ぐことができます
df_result.to_csv(output_filename, encoding='utf-8-sig', index=False)
print(f"結果を '{output_filename}' に出力しました。")